In [4]:
import requests
import pandas as pd
from pathlib import Path
import time


CELL_LINES_TSV = Path("/Users/bgadmin/Downloads/cellline_analyses_global_with_cnv_panel.tsv")
DOWNLOAD_DIR   = Path("/Users/bgadmin/Downloads/")

In [ ]:
import requests
import pandas as pd
from pathlib import Path
import time
df_cl = pd.read_csv(CELL_LINES_TSV, sep="\t", low_memory=False)

# Lignées uniquement (source_group == CellLine)
cl_ids = df_cl[df_cl["source_group"] == "CellLine"]["biosample_id"].unique()
print(f"Lignées cellulaires : {len(cl_ids)}")

# Récupère les Cellosaurus IDs via l'API Progenetix
records = []

for i, bid in enumerate(cl_ids):
    if i % 50 == 0:
        print(f"  {i}/{len(cl_ids)}...")

    url = f"https://progenetix.org/beacon/biosamples/{bid}/"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            records.append({"biosample_id": bid, "cellosaurus_id": None})
            continue

        data = resp.json()
        results = (data.get("response", {})
                      .get("resultSets", [{}])[0]
                      .get("results", [{}]))

        if not results:
            records.append({"biosample_id": bid, "cellosaurus_id": None})
            continue

        ext_refs = results[0].get("externalReferences", [])
        cellosaurus_id = None
        for ref in ext_refs:
            if ref.get("id", "").startswith("cellosaurus:"):
                cellosaurus_id = ref["id"].replace("cellosaurus:", "")
                break

        records.append({
            "biosample_id":   bid,
            "cellosaurus_id": cellosaurus_id,
            "cell_line_name": results[0].get("notes", ""),
        })

    except Exception as e:
        records.append({"biosample_id": bid, "cellosaurus_id": None})

    time.sleep(0.1)  # évite de surcharger l'API

df_bridge = pd.DataFrame(records)
print(f"\nTotal : {len(df_bridge)}")
print(f"Avec Cellosaurus ID : {df_bridge['cellosaurus_id'].notna().sum()}")
print(f"Sans Cellosaurus ID : {df_bridge['cellosaurus_id'].isna().sum()}")
print(df_bridge.head(10))

# Sauvegarde
df_bridge.to_csv(
    Path("/Users/bgadmin/Downloads/bridge_progenetix_cellosaurus.csv"),
    index=False
)
print("Saved: bridge_progenetix_cellosaurus.csv")

Lignées cellulaires : 5752
  0/5752...
  50/5752...
  100/5752...
  150/5752...
  200/5752...
  250/5752...
  300/5752...
  350/5752...
  400/5752...
  450/5752...
  500/5752...
  550/5752...
  600/5752...
  650/5752...
  700/5752...
  750/5752...
  800/5752...
  850/5752...
  900/5752...
  950/5752...
  1000/5752...
  1050/5752...
  1100/5752...
  1150/5752...
  1200/5752...
  1250/5752...
  1300/5752...
  1350/5752...
  1400/5752...
  1450/5752...
  1500/5752...
  1550/5752...
  1600/5752...
  1650/5752...
  1700/5752...
  1750/5752...
  1800/5752...
  1850/5752...
  1900/5752...
  1950/5752...
  2000/5752...
  2050/5752...
  2100/5752...
  2150/5752...
  2200/5752...
  2250/5752...
  2300/5752...
  2350/5752...
  2400/5752...
  2450/5752...
  2500/5752...
  2550/5752...
  2600/5752...
  2650/5752...
  2700/5752...
  2750/5752...
  2800/5752...
  2850/5752...
  2900/5752...
  2950/5752...
  3000/5752...
  3050/5752...
  3100/5752...
  3150/5752...
  3200/5752...
  3250/5752...
  3300

OSError: Cannot save file into a non-existent directory: '/Users/bgadmin/Downloads/ccle_multiomics'

In [6]:
# Crée le dossier
import os
os.makedirs("/Users/bgadmin/Downloads/ccle_multiomics", exist_ok=True)

# Sauvegarde df_bridge qui est déjà en mémoire
df_bridge.to_csv(
    "/Users/bgadmin/Downloads/ccle_multiomics/bridge_progenetix_cellosaurus.csv",
    index=False
)
print("Saved")
print(f"Total : {len(df_bridge)}")
print(f"Avec Cellosaurus ID : {df_bridge['cellosaurus_id'].notna().sum()}")
print(f"Sans Cellosaurus ID : {df_bridge['cellosaurus_id'].isna().sum()}")
print(df_bridge[df_bridge['cellosaurus_id'].notna()].head(10))

Saved
Total : 5752
Avec Cellosaurus ID : 5707
Sans Cellosaurus ID : 45
     biosample_id cellosaurus_id  \
0  pgxbs-kftvi9xs      CVCL_0139   
1  pgxbs-kftvi9xt      CVCL_0333   
2  pgxbs-kftvi9xv      CVCL_0371   
3  pgxbs-kftvi9xx      CVCL_1603   
4  pgxbs-kftvi9xy      CVCL_0099   
5  pgxbs-kftvi9y0      CVCL_0078   
6  pgxbs-kftvi9y1      CVCL_0076   
7  pgxbs-kftvi9xg      CVCL_0139   
8  pgxbs-kftvi9xi      CVCL_0333   
9  pgxbs-kftvi9xj      CVCL_0371   

                                      cell_line_name  
0             gastric adenocarcinoma [cell line AGS]  
1         Gastric adenocarcinoma [cell line Hs 746T]  
2  Signe ring gastric adenocarcinoma [cell line K...  
3  gastric tubular adenocarcinoma [cell line NCI-...  
4                Gastric carcinoma [cell line SNU-1]  
5                Gastric carcinoma [cell line SNU-5]  
6               Gastric carcinoma [cell line SNU-16]  
7             gastric adenocarcinoma [cell line AGS]  
8         Gastric adenocarcinoma [cel

In [10]:
import pandas as pd
from pathlib import Path

DOWNLOAD_DIR = Path("/Users/bgadmin/Downloads/")

# ── Load Progenetix → Cellosaurus bridge ──────────────────────
df_bridge = pd.read_csv(DOWNLOAD_DIR / "ccle_multiomics/bridge_progenetix_cellosaurus.csv")

# ── Load DepMap Model metadata ────────────────────────────────
model = pd.read_csv(DOWNLOAD_DIR / "Model.csv", low_memory=False)

# Clean RRID column to match Cellosaurus format (CVCL_XXXX)
model["cellosaurus_id"] = (
    model["RRID"]
    .str.replace("RRID:", "", regex=False)
    .str.strip()
)

print(f"Model.csv        : {len(model)} cell lines")
print(f"With RRID        : {model['cellosaurus_id'].notna().sum()}")
print(f"Sample RRID      : {model['cellosaurus_id'].dropna().head(5).tolist()}")

# ── Merge Progenetix bridge with DepMap Model ─────────────────
df_merged = df_bridge.merge(
    model[["ModelID", "CCLEName", "cellosaurus_id",
           "OncotreeLineage", "OncotreePrimaryDisease"]],
    on="cellosaurus_id",
    how="left"
)

print(f"\nAfter merge:")
print(f"  Total Progenetix cell lines  : {len(df_bridge)}")
print(f"  Mapped to DepMap ModelID     : {df_merged['ModelID'].notna().sum()}")
print(f"  Not mapped                   : {df_merged['ModelID'].isna().sum()}")
print()
print(df_merged[df_merged['ModelID'].notna()].head(10)[
    ["biosample_id", "cellosaurus_id", "ModelID", "CCLEName"]
])

# ── Save ──────────────────────────────────────────────────────
df_merged.to_csv(DOWNLOAD_DIR / "bridge_progenetix_depmap.csv", index=False)
print("\nSaved: bridge_progenetix_depmap.csv")

Model.csv        : 2154 cell lines
With RRID        : 1996
Sample RRID      : ['CVCL_0465', 'CVCL_0002', 'CVCL_0025', 'CVCL_0001', 'CVCL_2481']

After merge:
  Total Progenetix cell lines  : 5752
  Mapped to DepMap ModelID     : 11541
  Not mapped                   : 1287

     biosample_id cellosaurus_id     ModelID         CCLEName
0  pgxbs-kftvi9xs      CVCL_0139  ACH-000880      AGS_STOMACH
1  pgxbs-kftvi9xt      CVCL_0333  ACH-000616   HS746T_STOMACH
2  pgxbs-kftvi9xv      CVCL_0371  ACH-000793  KATOIII_STOMACH
3  pgxbs-kftvi9xx      CVCL_1603  ACH-000427   NCIN87_STOMACH
4  pgxbs-kftvi9xy      CVCL_0099  ACH-000932     SNU1_STOMACH
5  pgxbs-kftvi9y0      CVCL_0078  ACH-000303     SNU5_STOMACH
6  pgxbs-kftvi9y1      CVCL_0076  ACH-000581    SNU16_STOMACH
7  pgxbs-kftvi9xg      CVCL_0139  ACH-000880      AGS_STOMACH
8  pgxbs-kftvi9xi      CVCL_0333  ACH-000616   HS746T_STOMACH
9  pgxbs-kftvi9xj      CVCL_0371  ACH-000793  KATOIII_STOMACH

Saved: bridge_progenetix_depmap.csv


In [11]:
# ── Deduplicate — keep unique ModelIDs only ───────────────────
df_merged = pd.read_csv(DOWNLOAD_DIR / "bridge_progenetix_depmap.csv")

# How many unique DepMap ModelIDs do we have?
unique_models = (
    df_merged[df_merged["ModelID"].notna()]
    ["ModelID"]
    .unique()
)
print(f"Unique DepMap ModelIDs : {len(unique_models)}")

# Build a clean mapping: one row per biosample_id → ModelID
# Keep first match when multiple biosample_ids map to same ModelID
df_clean = (
    df_merged[df_merged["ModelID"].notna()]
    .drop_duplicates(subset=["biosample_id", "ModelID"])
    .copy()
)
print(f"Clean bridge rows      : {len(df_clean)}")

# How many biosample_ids map to more than one ModelID?
multi = df_clean.groupby("biosample_id")["ModelID"].nunique()
print(f"biosample_id with >1 ModelID : {(multi > 1).sum()}")
print(f"biosample_id with 1 ModelID  : {(multi == 1).sum()}")

# Save clean bridge
df_clean.to_csv(DOWNLOAD_DIR / "bridge_clean.csv", index=False)
print("\nSaved: bridge_clean.csv")
print(df_clean.head(5)[["biosample_id", "cellosaurus_id", "ModelID", "CCLEName"]])

Unique DepMap ModelIDs : 1398
Clean bridge rows      : 11541
biosample_id with >1 ModelID : 56
biosample_id with 1 ModelID  : 4409

Saved: bridge_clean.csv
     biosample_id cellosaurus_id     ModelID         CCLEName
0  pgxbs-kftvi9xs      CVCL_0139  ACH-000880      AGS_STOMACH
1  pgxbs-kftvi9xt      CVCL_0333  ACH-000616   HS746T_STOMACH
2  pgxbs-kftvi9xv      CVCL_0371  ACH-000793  KATOIII_STOMACH
3  pgxbs-kftvi9xx      CVCL_1603  ACH-000427   NCIN87_STOMACH
4  pgxbs-kftvi9xy      CVCL_0099  ACH-000932     SNU1_STOMACH


In [14]:
import pandas as pd
from pathlib import Path

DOWNLOAD_DIR = Path("/Users/bgadmin/Downloads/")

# ── Load bridge ───────────────────────────────────────────────
df_bridge = pd.read_csv(DOWNLOAD_DIR / "bridge_progenetix_depmap.csv")
our_model_ids = set(df_bridge["ModelID"].dropna().unique())
print(f"Our ModelIDs (from bridge) : {len(our_model_ids)}")

# ── Check RNA coverage ────────────────────────────────────────
print("\nReading RNA header...")
rna_path = DOWNLOAD_DIR / "ccle_multiomics/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
rna_cols  = pd.read_csv(rna_path, nrows=0).columns.tolist()
# First column is gene index, rest are ModelIDs
rna_model_ids = set(rna_cols[1:])
rna_available = our_model_ids & rna_model_ids
print(f"Total cell lines in RNA file : {len(rna_model_ids)}")
print(f"Our cell lines with RNA      : {len(rna_available)}")

# ── Check Methylation coverage ────────────────────────────────
print("\nReading Methylation header...")
meth_path = DOWNLOAD_DIR / "ccle_multiomics/CCLE_RRBS_TSS_1kb_20180614.txt"
meth_cols  = pd.read_csv(meth_path, sep="\t", nrows=0).columns.tolist()
print(f"Sample meth columns : {meth_cols[:10]}")

# Methylation file uses CCLEName not ModelID
# Need to map CCLEName → ModelID via Model.csv
model = pd.read_csv(DOWNLOAD_DIR / "Model.csv", low_memory=False)
model["cellosaurus_id"] = (
    model["RRID"]
    .str.replace("RRID:", "", regex=False)
    .str.strip()
)

# Build CCLEName → ModelID lookup
ccle_to_model = model.set_index("CCLEName")["ModelID"].to_dict()

meth_ccle_ids  = set(meth_cols)
meth_model_ids = set(
    ccle_to_model[c] for c in meth_ccle_ids if c in ccle_to_model
)
meth_available = our_model_ids & meth_model_ids
print(f"Total cell lines in Meth file : {len(meth_ccle_ids)}")
print(f"Our cell lines with Meth      : {len(meth_available)}")

# ── Three omics intersection ──────────────────────────────────
all_three = rna_available & meth_available
print(f"\nCell lines with CNV + RNA + Meth : {len(all_three)}")
print(f"Cell lines with CNV + RNA only   : {len(rna_available - meth_available)}")
print(f"Cell lines with CNV only         : {len(our_model_ids - rna_available - meth_available)}")

Our ModelIDs (from bridge) : 1398

Reading RNA header...
Total cell lines in RNA file : 19220
Our cell lines with RNA      : 0

Reading Methylation header...
Sample meth columns : ['TSS_id', 'gene', 'chr', 'fpos', 'tpos', 'strand', 'avg_coverage', 'DMS53_LUNG', 'SW1116_LARGE_INTESTINE', 'P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE']
Total cell lines in Meth file : 850
Our cell lines with Meth      : 827

Cell lines with CNV + RNA + Meth : 0
Cell lines with CNV + RNA only   : 0
Cell lines with CNV only         : 571


In [17]:
# ── Fix RNA — rows are cell lines, not columns ────────────────
print("Reading RNA ModelID column...")
rna_model_col = pd.read_csv(rna_path, usecols=["ModelID"])
rna_model_ids = set(rna_model_col["ModelID"].dropna().unique())

print(f"Total cell lines in RNA file : {len(rna_model_ids)}")
print(f"Sample RNA ModelIDs : {list(rna_model_ids)[:5]}")

rna_available = our_model_ids & rna_model_ids
print(f"Our cell lines with RNA      : {len(rna_available)}")

# ── Fix Methylation — use CCLEName directly ───────────────────
meth_data_cols = [c for c in meth_cols
                  if c not in ["TSS_id", "gene", "chr",
                               "fpos", "tpos", "strand",
                               "avg_coverage"]]

# Map meth CCLEName → ModelID
ccle_to_model  = model.set_index("CCLEName")["ModelID"].to_dict()
meth_model_ids = set(
    ccle_to_model[c] for c in meth_data_cols if c in ccle_to_model
)
meth_available = our_model_ids & meth_model_ids
print(f"\nOur cell lines with Meth     : {len(meth_available)}")

# ── Three omics intersection ──────────────────────────────────
all_three = rna_available & meth_available
rna_only  = rna_available - meth_available
cnv_only  = our_model_ids - rna_available - meth_available

print(f"\nCell lines with CNV + RNA + Meth : {len(all_three)}")
print(f"Cell lines with CNV + RNA only   : {len(rna_only)}")
print(f"Cell lines with CNV only         : {len(cnv_only)}")

# ── Save coverage summary ─────────────────────────────────────
coverage = pd.DataFrame({
    "ModelID": list(our_model_ids),
    "has_rna":  [m in rna_available  for m in our_model_ids],
    "has_meth": [m in meth_available for m in our_model_ids],
})
coverage["has_all_three"] = coverage["has_rna"] & coverage["has_meth"]
coverage.to_csv(DOWNLOAD_DIR / "coverage_summary.csv", index=False)
print("\nSaved: coverage_summary.csv")

Reading RNA ModelID column...
Total cell lines in RNA file : 1719
Sample RNA ModelIDs : ['ACH-001541', 'ACH-000563', 'ACH-000346', 'ACH-001765', 'ACH-000484']
Our cell lines with RNA      : 1198

Our cell lines with Meth     : 827

Cell lines with CNV + RNA + Meth : 818
Cell lines with CNV + RNA only   : 380
Cell lines with CNV only         : 191

Saved: coverage_summary.csv


In [18]:
import h5py
from pathlib import Path

MOFA_MODEL = Path("/Users/bgadmin/Downloads/multiomics_preproc/mofa_model.hdf5")

with h5py.File(MOFA_MODEL, "r") as f:
    features_rna  = [x.decode() for x in f["features/RNA"][:]]
    features_meth = [x.decode() for x in f["features/Meth"][:]]
    features_cnv  = [x.decode() for x in f["features/CNV"][:]]

print(f"MOFA RNA features  : {len(features_rna)}")
print(f"MOFA Meth features : {len(features_meth)}")
print(f"MOFA CNV features  : {len(features_cnv)}")
print(f"\nSample RNA  : {features_rna[:5]}")
print(f"Sample Meth : {features_meth[:5]}")
print(f"Sample CNV  : {features_cnv[:5]}")

MOFA RNA features  : 1301
MOFA Meth features : 1171
MOFA CNV features  : 5128

Sample RNA  : ['ARID4B_RNA', 'PICALM_RNA', 'HOXC11_RNA', 'FANCE_RNA', 'SMC1A_RNA']
Sample Meth : ['ARID4B_Meth', 'PICALM_Meth', 'EPCAM_Meth', 'ERG_Meth', 'FOXR1_Meth']
Sample CNV  : ['A1CF__dup_CNV', 'ABCB1__dup_CNV', 'ABCC3__dup_CNV', 'ABI1__dup_CNV', 'ABL1__dup_CNV']


In [19]:
import pandas as pd
import h5py
from pathlib import Path

DOWNLOAD_DIR = Path("/Users/bgadmin/Downloads/")
MOFA_MODEL   = Path("/Users/bgadmin/Downloads/multiomics_preproc/mofa_model.hdf5")

# ── Load MOFA features ────────────────────────────────────────
with h5py.File(MOFA_MODEL, "r") as f:
    features_rna  = [x.decode() for x in f["features/RNA"][:]]
    features_meth = [x.decode() for x in f["features/Meth"][:]]

# Extract gene names from MOFA feature names
# 'ARID4B_RNA' → 'ARID4B'
# 'ARID4B_Meth' → 'ARID4B'
mofa_rna_genes  = set(f.replace("_RNA",  "") for f in features_rna)
mofa_meth_genes = set(f.replace("_Meth", "") for f in features_meth)

print(f"MOFA RNA genes  : {len(mofa_rna_genes)}")
print(f"MOFA Meth genes : {len(mofa_meth_genes)}")

# ── Check RNA gene format in DepMap file ──────────────────────
# DepMap RNA columns look like: 'TSPAN6 (7105)'
# We need to extract just the gene name
rna_path = DOWNLOAD_DIR / "ccle_multiomics/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
rna_header = pd.read_csv(rna_path, nrows=0).columns.tolist()

# Gene columns start after metadata columns
meta_cols = ["Unnamed: 0", "SequencingID", "ModelConditionID",
             "ModelID", "IsDefaultEntryForMC", "IsDefaultEntryForModel"]
gene_cols = [c for c in rna_header if c not in meta_cols]

# Extract gene names: 'TSPAN6 (7105)' → 'TSPAN6'
depmap_rna_genes = set(c.split(" (")[0] for c in gene_cols)

print(f"\nDepMap RNA genes total   : {len(depmap_rna_genes)}")
rna_overlap = mofa_rna_genes & depmap_rna_genes
print(f"Overlap MOFA ∩ DepMap RNA : {len(rna_overlap)}")
print(f"MOFA RNA genes not in DepMap : {len(mofa_rna_genes - depmap_rna_genes)}")
print(f"Missing : {list(mofa_rna_genes - depmap_rna_genes)[:10]}")

# ── Check Meth gene format in CCLE file ───────────────────────
meth_path = DOWNLOAD_DIR / "ccle_multiomics/CCLE_RRBS_TSS_1kb_20180614.txt"
meth_df_head = pd.read_csv(meth_path, sep="\t", nrows=5)
print(f"\nMeth file columns : {meth_df_head.columns.tolist()[:8]}")
print(f"\nMeth file sample rows :")
print(meth_df_head[["TSS_id", "gene"]].head(10))

# Gene names in meth file
meth_genes_full = pd.read_csv(meth_path, sep="\t",
                               usecols=["gene"])["gene"].dropna().unique()
depmap_meth_genes = set(meth_genes_full)
print(f"\nDepMap Meth genes total    : {len(depmap_meth_genes)}")
meth_overlap = mofa_meth_genes & depmap_meth_genes
print(f"Overlap MOFA ∩ DepMap Meth : {len(meth_overlap)}")
print(f"MOFA Meth genes not in DepMap : {len(mofa_meth_genes - depmap_meth_genes)}")
print(f"Missing : {list(mofa_meth_genes - depmap_meth_genes)[:10]}")

MOFA RNA genes  : 1301
MOFA Meth genes : 1171

DepMap RNA genes total   : 19215
Overlap MOFA ∩ DepMap RNA : 1292
MOFA RNA genes not in DepMap : 9
Missing : ['HMGN2P46', 'CDC50A', 'TERC', 'H3P6', 'TMSB4XP8', 'NBEAP1', 'MALAT1', 'RNF217-AS1', 'MDS2']

Meth file columns : ['TSS_id', 'gene', 'chr', 'fpos', 'tpos', 'strand', 'avg_coverage', 'DMS53_LUNG']

Meth file sample rows :
                         TSS_id          gene
0  LOC100288069_1_714068_715068  LOC100288069
1     LINC01128_1_761970_762970     LINC01128
2     LINC01128_1_762177_763177     LINC01128
3  LOC100130417_1_855072_856072  LOC100130417
4        SAMD11_1_860120_861120        SAMD11

DepMap Meth genes total    : 16494
Overlap MOFA ∩ DepMap Meth : 982
MOFA Meth genes not in DepMap : 189
Missing : ['WAS', 'MACC1', 'CD36', 'EIF3E', 'PIK3C2B', 'APH1A', 'TNFRSF17', 'ZNF217', 'LPP', 'APOBEC3B']


In [20]:
import pandas as pd
import numpy as np
import h5py
from pathlib import Path

DOWNLOAD_DIR = Path("/Users/bgadmin/Downloads/")
MOFA_MODEL   = Path("/Users/bgadmin/Downloads/multiomics_preproc/mofa_model.hdf5")

# ── Load MOFA features ────────────────────────────────────────
with h5py.File(MOFA_MODEL, "r") as f:
    features_rna  = [x.decode() for x in f["features/RNA"][:]]
    features_meth = [x.decode() for x in f["features/Meth"][:]]

mofa_rna_genes  = [f.replace("_RNA",  "") for f in features_rna]
mofa_meth_genes = [f.replace("_Meth", "") for f in features_meth]

# ── Load coverage summary and bridge ─────────────────────────
coverage = pd.read_csv(DOWNLOAD_DIR / "coverage_summary.csv")
bridge   = pd.read_csv(DOWNLOAD_DIR / "bridge_progenetix_depmap.csv")
model    = pd.read_csv(DOWNLOAD_DIR / "Model.csv", low_memory=False)

# Cell lines with all three omics
cl_all_three = coverage[coverage["has_all_three"]]["ModelID"].tolist()
cl_rna_only  = coverage[coverage["has_rna"] & ~coverage["has_meth"]]["ModelID"].tolist()
print(f"Cell lines with CNV + RNA + Meth : {len(cl_all_three)}")
print(f"Cell lines with CNV + RNA only   : {len(cl_rna_only)}")

Cell lines with CNV + RNA + Meth : 818
Cell lines with CNV + RNA only   : 380


In [26]:
# ── Extract RNA data for our cell lines ───────────────────────
print("Extracting RNA data...")

rna_path = DOWNLOAD_DIR / "ccle_multiomics/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"

# Build mapping: gene name → column name in DepMap file
rna_header   = pd.read_csv(rna_path, nrows=0).columns.tolist()
meta_cols    = ["Unnamed: 0", "SequencingID", "ModelConditionID",
                "ModelID", "IsDefaultEntryForMC", "IsDefaultEntryForModel"]
gene_cols    = [c for c in rna_header if c not in meta_cols]

# Map MOFA gene name → DepMap column name
# 'ARID4B' → 'ARID4B (8165)'
gene_to_col  = {c.split(" (")[0]: c for c in gene_cols}
mofa_rna_cols = [gene_to_col[g] for g in mofa_rna_genes if g in gene_to_col]
missing_rna   = [g for g in mofa_rna_genes if g not in gene_to_col]
print(f"  MOFA RNA features found in DepMap : {len(mofa_rna_cols)}/1301")
print(f"  Missing                           : {len(missing_rna)}")

# Read only the columns we need
cols_to_read = ["ModelID", "IsDefaultEntryForModel"] + mofa_rna_cols
df_rna_raw   = pd.read_csv(rna_path, usecols=cols_to_read, low_memory=False)

# Keep only default entry per model and our cell lines
df_rna_raw = df_rna_raw[df_rna_raw["IsDefaultEntryForModel"] == "Yes"].copy()
df_rna_raw   = df_rna_raw[df_rna_raw["ModelID"].isin(
    set(cl_all_three + cl_rna_only)
)].copy()

# Rename columns: 'ARID4B (8165)' → 'ARID4B_RNA'
rename_map   = {c: c.split(" (")[0] + "_RNA" for c in mofa_rna_cols}
df_rna_raw   = df_rna_raw.rename(columns=rename_map)
df_rna_raw   = df_rna_raw.set_index("ModelID").drop(
    columns=["IsDefaultEntryForModel"]
)

print(f"  RNA matrix shape : {df_rna_raw.shape}")
print(f"  Sample : {df_rna_raw.iloc[:3, :5]}")

# Save
df_rna_raw.to_parquet(DOWNLOAD_DIR / "celllines_rna_mofa_features.parquet")
print("  Saved: celllines_rna_mofa_features.parquet")

Extracting RNA data...
  MOFA RNA features found in DepMap : 1292/1301
  Missing                           : 9
  RNA matrix shape : (1198, 1292)
  Sample :             RAD52_RNA  LASP1_RNA  RECQL_RNA  UPF1_RNA  HOXA11_RNA
ModelID                                                          
ACH-001113   3.706387   5.417643   6.858205  4.575215    0.313349
ACH-002438   2.691789   7.899043   6.399429  4.064736    5.674195
ACH-000242   3.147041   7.002579   5.015533  5.373485    2.800531
  Saved: celllines_rna_mofa_features.parquet


In [24]:
# ── Extract Methylation data for our cell lines ───────────────
print("Extracting Methylation data...")

meth_path = DOWNLOAD_DIR / "ccle_multiomics/CCLE_RRBS_TSS_1kb_20180614.txt"

ccle_to_model = model.set_index("CCLEName")["ModelID"].to_dict()

print("  Reading methylation file (may take 2-3 min)...")
df_meth_raw = pd.read_csv(meth_path, sep="\t", low_memory=False)
print(f"  Raw shape : {df_meth_raw.shape}")

# Filter rows to MOFA meth genes only
mofa_meth_set = set(mofa_meth_genes)
df_meth_raw   = df_meth_raw[df_meth_raw["gene"].isin(mofa_meth_set)].copy()
print(f"  After gene filter : {df_meth_raw.shape}")

# Cell line columns only
meta_meth_cols = ["TSS_id", "gene", "chr", "fpos", "tpos",
                  "strand", "avg_coverage"]
cl_cols        = [c for c in df_meth_raw.columns if c not in meta_meth_cols]

# Keep only our cell lines
our_ccle_names = set()
for mid in cl_all_three:
    row = model[model["ModelID"] == mid]
    if not row.empty and pd.notna(row["CCLEName"].values[0]):
        our_ccle_names.add(row["CCLEName"].values[0])

cl_cols_keep = [c for c in cl_cols if c in our_ccle_names]
print(f"  Cell line columns kept : {len(cl_cols_keep)}")

df_meth_filt = df_meth_raw[["gene"] + cl_cols_keep].copy()

# Convert to numeric — replace any string NAs with np.nan
for col in cl_cols_keep:
    df_meth_filt[col] = pd.to_numeric(df_meth_filt[col], errors="coerce")

print(f"  Non-numeric values converted to NaN")

# Aggregate: mean methylation per gene per cell line
df_meth_agg = (
    df_meth_filt
    .groupby("gene")[cl_cols_keep]
    .mean()
)
print(f"  After aggregation : {df_meth_agg.shape}")

# Transpose: rows = cell lines, columns = genes
df_meth_T = df_meth_agg.T.copy()
df_meth_T.index.name = "CCLEName"

# Add ModelID
df_meth_T["ModelID"] = df_meth_T.index.map(ccle_to_model)
df_meth_T = df_meth_T.set_index("ModelID")

# Rename columns: 'ARID4B' → 'ARID4B_Meth'
df_meth_T.columns = [c + "_Meth" for c in df_meth_T.columns]

print(f"  Final meth matrix shape : {df_meth_T.shape}")

# Save
df_meth_T.to_parquet(DOWNLOAD_DIR / "celllines_meth_mofa_features.parquet")
print("  Saved: celllines_meth_mofa_features.parquet")

Extracting Methylation data...
  Reading methylation file (may take 2-3 min)...
  Raw shape : (20192, 850)
  After gene filter : (1357, 850)
  Cell line columns kept : 818
  Non-numeric values converted to NaN
  After aggregation : (982, 818)
  Final meth matrix shape : (818, 982)
  Saved: celllines_meth_mofa_features.parquet
